In [0]:
select * from ecommerce_dev.gold.olist_datacube;

## 1 | Total revenue by month

In [0]:
select order_month as month_number , sum(total_payment_value)  as revenue_per_month
from ecommerce_dev.gold.olist_datacube
group by order_month
order by order_month asc;

## 2 | Average Order Value (AOV)

In [0]:
with total_rev as(
  select sum(total_payment_value) as total from ecommerce_dev.gold.olist_datacube
),
no_of_orders as(
  select count(distinct order_id) as total_orders from ecommerce_dev.gold.olist_datacube
)
select total_rev.total/no_of_orders.total_orders as avg_order_value from total_rev, no_of_orders;



## 3 | Top 10 products by revenue


In [0]:
select product_id , product_category_name as product_name, sum(total_payment_value)  as total_revenue
from ecommerce_dev.gold.olist_datacube
group by product_id ,product_category_name
order by sum(total_payment_value) desc
limit 10;

## 4 | Top 10 sellers by revenue

In [0]:
select seller_id , sum(total_payment_value)  as total_revenue
from ecommerce_dev.gold.olist_datacube
group by seller_id
order by sum(total_payment_value) desc
limit 10;

## 5 | Orders by customer city and state

In [0]:
select customer_state , customer_city ,sum(total_payment_value) as total_revenue
from ecommerce_dev.gold.olist_datacube
group by customer_state , customer_city
order by customer_state , customer_city;

## 6 | New vs returning customers (monthly)

In [0]:

WITH customer_first_purchase AS (
    SELECT
        customer_id,
        MIN(order_purchase_timestamp) AS first_purchase_date
    FROM ecommerce_dev.gold.olist_datacube
    GROUP BY customer_id
),
order_with_flag as (
    select o.order_id , o.customer_id ,date_trunc('month' , o.order_purchase_timestamp) as order_month , date_trunc('month' , c.first_purchase_date) as first_purchase_month
    from ecommerce_dev.gold.olist_datacube as  o
    join customer_first_purchase as c
    on o.customer_id = c.customer_id
)

select order_month ,
count(distinct 
            case 
                when order_month = first_purchase_month then customer_id
            end) as new_customers,
count(distinct 
            case 
                when order_month > first_purchase_month then customer_id
            end) as existing_customers
from order_with_flag
group by order_month
order by order_month asc
   

## 7 | Customer Lifetime Value (CLV)

In [0]:
with tr as (
  select sum(total_payment_value) as total_revenue, 
         count(distinct order_id) as total_orders, 
         count(distinct customer_id) as total_customers, 
         datediff(year, min(order_purchase_timestamp), max(order_delivered_customer_date)) as active_years 
  from ecommerce_dev.gold.olist_datacube
)
select total_revenue/total_orders as apv  , total_orders/total_customers as pf ,
active_years/total_customers  as cls , (apv*pf*cls) as customer_lifetime_value from tr;

## 8 | Order fulfillment rate


In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
),
filtered as (
  select * from dup where rn = 1
),
delivered_count as (
  select count(*) as isdel_count from filtered where is_delivered = true
),
total_count as (
  select count(*) as total from filtered
)
select delivered_count.isdel_count*100 / total_count.total as order_fullfillment_rate
from delivered_count, total_count;

## 9 | Late delivery percentage

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
),
filtered as (
  select * from dup where rn = 1
),
late_del_count as (
  select count(*) as isdel_count from filtered where is_late_delivery = false
),
total_count as (
  select count(*) as total from filtered
)
select late_del_count.isdel_count*100 / total_count.total as late_delivery_percentage
from late_del_count, total_count;

## 10 | Average delivery time (days)  --region**

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
),
dup2 as (
  select * , datediff(day, order_purchase_timestamp , order_delivered_customer_date) as del_day from dup
)
select customer_state ,customer_city ,avg(del_day) as avg_delivery_time_in_day from dup2
group by customer_state ,customer_city
order by customer_state, customer_city;


## 11 | Payment method distribution

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
)
select payment_type , count(*) 
from dup
where payment_type is not null
group by payment_type
order by payment_type;


## 12 | Average review score by product category

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
)
select product_category_name , avg(review_score) as avg_review_score from dup
where product_category_name is not null
group by product_category_name

## 13 | Average review score by seller

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
)
select seller_id , avg(review_score) as avg_review_score from dup
where seller_id is not null
group by seller_id
order by seller_id

## 14 | Percentage of low-rated orders (rating < 3)

In [0]:
with dup as (
  select *, row_number() over (partition by order_id order by order_purchase_timestamp) as rn 
  from ecommerce_dev.gold.olist_datacube 
),
neg as (
  select count(*) as n_count from dup 
  where review_category = "negative"
),
tot as(
  select  count(*) as t_count from dup
)
select neg.n_count*100/tot.t_count as negative_review_percentage from neg, tot